# 04. Финансовая оценка, интерпретация, стабильность (этап 4)
Test (out-of-time) открывается **один раз** — внутри `P.step4_evaluation()`. Здесь же: Gini и полнота при фиксированной доле проверок, прибыль портфеля (LGD = 45%), SHAP и разбор трёх наблюдений, PSI, сохранение `model.pkl`.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

In [ ]:
r4 = P.ensure("step4")
print("движок:", r4["engine"], "| метод объяснений:", r4["shap_method"], "| test:", r4["test_n"], "компаний,", r4["test_bankrupt"], "банкротств")

## 1. Метрики варианта на out-of-time test

In [ ]:
def row(d, name):
    return {"модель": name, "Gini": d["gini"], "Gini ДИ95": tuple(round(x, 3) for x in d["gini_ci"]), "KS": d["ks"], "PR-AUC": d["pr_auc"],
            **{f"полнота@{s}%": d[f"recall@{s}%"] for s in (5, 10, 20)}, "Brier (калибр.)": d["brier_calibrated"]}
pd.DataFrame([row(r4["base"], "скоринговая карта"), row(r4["boost"], "бустинг")])

In [ ]:
print("Gini(бустинг) − Gini(карта):", r4["gini_diff"])

In [ ]:
P.show_fig("fig_calibration_test.png")

## 2. Перевод качества в деньги
Одобряем компанию, если PD < порога. Доход с хорошей одобренной компании — `margin × EAD` (маржа 8% — **допущение**, в данных нет сумм), потери с одобренной обанкротившейся — `LGD × EAD`, LGD = 45%. Порог выбран на validation по максимуму **ожидаемой** прибыли по калиброванным PD; на test считаем реализованную прибыль.

In [ ]:
f = r4["finance"]
tab = pd.DataFrame({k: {x: f[k][x] for x in ("approve_rate", "good_approved", "bad_approved", "income", "loss", "profit", "profit_per_exposure")}
                    for k in ("all_approved", "boosting", "baseline", "oracle")}).T
print("порог PD (бустинг):", round(f["threshold_pd"], 4), "| порог PD (карта):", round(f["threshold_pd_baseline"], 4))
tab

In [ ]:
pd.DataFrame(f["sensitivity"])

In [ ]:
P.show_fig("fig_finance.png")

## 3. SHAP: summary plot и разбор трёх наблюдений

In [ ]:
P.show_fig("fig_shap_summary.png")

In [ ]:
for cs in r4["cases"]:
    print(f'--- {cs["title"]} | PD={cs["pd_calibrated"]:.1%} | факт: {"банкрот" if cs["target"] else "не банкрот"} | решение: {cs["decision"]}')
    print(pd.DataFrame(cs["top"]).to_string(index=False))
P.show_fig("fig_shap_cases.png")

## 4. Стабильность: PSI train → validation / test
PSI > 0,25 — признак нестабилен; 0,10–0,25 — умеренный сдвиг.

In [ ]:
pd.DataFrame(r4["psi"])

In [ ]:
P.show_fig("fig_psi.png")

## 5. Сохранённая модель и `predict(df)`

In [ ]:
import pandas as pd
from src.predict import predict
raw = pd.read_csv("data/data.csv")
print(predict(raw.tail(5), return_decision=True))
print("predict(df) совпадает с оценкой в пайплайне:", r4["predict_check_ok"])

**Вывод.** На test (1023 компании, 19 банкротов) Gini бустинга 0,935, карты 0,931; разница 0,004, доверительный интервал [−0,016; 0,029] — модели статистически неразличимы. Полнота при проверке 10% компаний: 79% у бустинга и 84% у карты. Стратегия с порогом PD даёт у бустинга 7,42% прибыли от экспозиции против 7,02% при «одобрить всех» (маржа 8% — допущение). По SHAP главные драйверы — retained_self_financing и показатели рентабельности; PSI скора 0,031 (стабилен), из топ-5 признаков нестабилен только Quick Ratio (PSI 0,33).